# K-Nearest Neighbor (KNN) — Step-by-Step Solution

**Problem:** Predict whether a student passes an exam using KNN (K=3 and K=5)  
**Features:** No. Programming Languages used, No. delivered assignments, GPA  
**Label:** Passed Exam (Yes / No)

---
## Step 1 — Import NumPy

In [1]:
import numpy as np

---
## Step 2 — Define Training Data

| ID | Languages | Assignments | GPA | Passed |
|----|-----------|-------------|-----|--------|
| 1  | 3         | 5           | 3.1 | Yes    |
| 2  | 3         | 5           | 2.0 | No     |
| 3  | 2         | 7           | 3.5 | Yes    |
| 4  | 4         | 9           | 2.5 | Yes    |
| 5  | 3         | 11          | 3.9 | No     |
| 6  | 2         | 3           | 2.9 | No     |
| 7  | 3         | 3           | 1.9 | No     |
| 8  | 4         | 9           | 3.2 | Yes    |

In [2]:
# Features: [No. Programming Languages, No. Assignments, GPA]
X_train = np.array([
    [3, 5,  3.1],  # ID 1
    [3, 5,  2.0],  # ID 2
    [2, 7,  3.5],  # ID 3
    [4, 9,  2.5],  # ID 4
    [3, 11, 3.9],  # ID 5
    [2, 3,  2.9],  # ID 6
    [3, 3,  1.9],  # ID 7
    [4, 9,  3.2],  # ID 8
])

# Labels: 1 = Yes (Passed), 0 = No (Failed)
y_train = np.array([1, 0, 1, 1, 0, 0, 0, 1])
label_map = {1: 'Yes', 0: 'No'}

print('Training features shape:', X_train.shape)
print('Training labels:', y_train)

Training features shape: (8, 3)
Training labels: [1 0 1 1 0 0 0 1]


---
## Step 3 — Define Test Data (Students to Predict)

| ID | Languages | Assignments | GPA |
|----|-----------|-------------|-----|
| 9  | 2         | 2           | 3.0 |
| 10 | 3         | 4           | 4.0 |
| 11 | 4         | 6           | 2.0 |
| 12 | 2         | 5           | 3.5 |

In [3]:
X_test = np.array([
    [2, 2, 3.0],  # ID 9
    [3, 4, 4.0],  # ID 10
    [4, 6, 2.0],  # ID 11
    [2, 5, 3.5],  # ID 12
])

test_ids = [9, 10, 11, 12]
print('Test features shape:', X_test.shape)

Test features shape: (4, 3)


---
## Step 4 — Euclidean Distance Function

The Euclidean distance between two points **p** and **q** in n-dimensional space:

$$d(p, q) = \sqrt{\sum_{i=1}^{n}(p_i - q_i)^2}$$

In [4]:
def euclidean_distance(x1, x2):
    """Compute Euclidean distance between two feature vectors."""
    return np.sqrt(np.sum((x1 - x2) ** 2))

# Quick test
example_dist = euclidean_distance(X_test[0], X_train[0])
print(f'Distance between ID 9 and ID 1: {example_dist:.4f}')

Distance between ID 9 and ID 1: 3.1639


---
## Step 5 — Compute All Distances for Each Test Point

For every test student, compute the distance to **all 8 training students**.

In [5]:
print('=== Distance Matrix (Test vs Training) ===\n')
print(f'{"":<8}', end='')
for i in range(1, 9):
    print(f'ID {i:<6}', end='')
print()
print('-' * 75)

distance_matrix = np.zeros((len(X_test), len(X_train)))

for i, test_point in enumerate(X_test):
    print(f'ID {test_ids[i]:<4}  ', end='')
    for j, train_point in enumerate(X_train):
        d = euclidean_distance(test_point, train_point)
        distance_matrix[i, j] = d
        print(f'{d:<9.4f}', end='')
    print()

print('\ndistance_matrix shape:', distance_matrix.shape)

=== Distance Matrix (Test vs Training) ===

        ID 1     ID 2     ID 3     ID 4     ID 5     ID 6     ID 7     ID 8     
---------------------------------------------------------------------------
ID 9     3.1639   3.3166   5.0249   7.2973   9.1000   1.0050   1.7916   7.2829   
ID 10    1.3454   2.2361   3.2016   5.3151   7.0007   1.7916   2.3259   5.1614   
ID 11    1.7916   1.4142   2.6926   3.0414   5.4415   3.7162   3.1639   3.2311   
ID 12    1.0770   1.8028   2.0000   4.5826   6.0959   2.0881   2.7495   4.4822   

distance_matrix shape: (4, 8)


---
## Step 6 — KNN Predict Function

For each test point:
1. **Sort** distances → get indices of K nearest neighbours (`np.argsort`)
2. **Fetch** their labels
3. **Vote** → majority class wins (`np.bincount` + `np.argmax`)

In [6]:
def knn_predict(X_train, y_train, X_test, k):
    """
    Predict class labels for X_test using K-Nearest Neighbors.

    Parameters
    ----------
    X_train : ndarray, shape (n_train, n_features)
    y_train : ndarray, shape (n_train,)  — integer labels
    X_test  : ndarray, shape (n_test,  n_features)
    k       : int — number of neighbours

    Returns
    -------
    List of dicts with keys: prediction, distances, neighbour_indices
    """
    results = []

    for test_point in X_test:
        # ── Step A: compute distances to all training points ──────────────
        distances = np.array([
            euclidean_distance(test_point, tp) for tp in X_train
        ])

        # ── Step B: sort and pick K nearest ───────────────────────────────
        k_indices = np.argsort(distances)[:k]        # indices of k smallest
        k_labels  = y_train[k_indices]               # their class labels
        k_distances = distances[k_indices]           # their distances

        # ── Step C: majority vote ──────────────────────────────────────────
        counts     = np.bincount(k_labels)           # count 0s and 1s
        prediction = np.argmax(counts)               # class with most votes

        results.append({
            'prediction':       prediction,
            'k_indices':        k_indices,           # 0-based
            'k_distances':      k_distances,
            'k_labels':         k_labels,
            'vote_counts':      counts,
        })

    return results

print('knn_predict() defined successfully.')

knn_predict() defined successfully.


---
## Step 7 — Predict with K = 3

In [7]:
k = 3
results_k3 = knn_predict(X_train, y_train, X_test, k)

print(f'{'='*65}')
print(f'  KNN Results — K = {k}')
print(f'{'='*65}')

for i, r in enumerate(results_k3):
    sid         = test_ids[i]
    neighbour_ids   = r['k_indices'] + 1   # convert to 1-based training IDs
    neighbour_dists = r['k_distances']
    neighbour_labels= [label_map[l] for l in r['k_labels']]
    vote_yes    = int(r['vote_counts'][1]) if len(r['vote_counts']) > 1 else 0
    vote_no     = int(r['vote_counts'][0])
    prediction  = label_map[r['prediction']]

    print(f'\nStudent ID {sid}')
    print(f'  Neighbours : {list(neighbour_ids)}')
    print(f'  Distances  : {[round(d,4) for d in neighbour_dists]}')
    print(f'  Labels     : {neighbour_labels}')
    print(f'  Votes      : Yes={vote_yes}, No={vote_no}')
    print(f'  ➜ Prediction: {prediction}')

  KNN Results — K = 3

Student ID 9
  Neighbours : [np.int64(6), np.int64(7), np.int64(1)]
  Distances  : [np.float64(1.005), np.float64(1.7916), np.float64(3.1639)]
  Labels     : ['No', 'No', 'Yes']
  Votes      : Yes=1, No=2
  ➜ Prediction: No

Student ID 10
  Neighbours : [np.int64(1), np.int64(6), np.int64(2)]
  Distances  : [np.float64(1.3454), np.float64(1.7916), np.float64(2.2361)]
  Labels     : ['Yes', 'No', 'No']
  Votes      : Yes=1, No=2
  ➜ Prediction: No

Student ID 11
  Neighbours : [np.int64(2), np.int64(1), np.int64(3)]
  Distances  : [np.float64(1.4142), np.float64(1.7916), np.float64(2.6926)]
  Labels     : ['No', 'Yes', 'Yes']
  Votes      : Yes=2, No=1
  ➜ Prediction: Yes

Student ID 12
  Neighbours : [np.int64(1), np.int64(2), np.int64(3)]
  Distances  : [np.float64(1.077), np.float64(1.8028), np.float64(2.0)]
  Labels     : ['Yes', 'No', 'Yes']
  Votes      : Yes=2, No=1
  ➜ Prediction: Yes


---
## Step 8 — Predict with K = 5

In [8]:
k = 5
results_k5 = knn_predict(X_train, y_train, X_test, k)

print(f'{'='*65}')
print(f'  KNN Results — K = {k}')
print(f'{'='*65}')

for i, r in enumerate(results_k5):
    sid         = test_ids[i]
    neighbour_ids   = r['k_indices'] + 1
    neighbour_dists = r['k_distances']
    neighbour_labels= [label_map[l] for l in r['k_labels']]
    vote_yes    = int(r['vote_counts'][1]) if len(r['vote_counts']) > 1 else 0
    vote_no     = int(r['vote_counts'][0])
    prediction  = label_map[r['prediction']]

    print(f'\nStudent ID {sid}')
    print(f'  Neighbours : {list(neighbour_ids)}')
    print(f'  Distances  : {[round(d,4) for d in neighbour_dists]}')
    print(f'  Labels     : {neighbour_labels}')
    print(f'  Votes      : Yes={vote_yes}, No={vote_no}')
    print(f'  ➜ Prediction: {prediction}')

  KNN Results — K = 5

Student ID 9
  Neighbours : [np.int64(6), np.int64(7), np.int64(1), np.int64(2), np.int64(3)]
  Distances  : [np.float64(1.005), np.float64(1.7916), np.float64(3.1639), np.float64(3.3166), np.float64(5.0249)]
  Labels     : ['No', 'No', 'Yes', 'No', 'Yes']
  Votes      : Yes=2, No=3
  ➜ Prediction: No

Student ID 10
  Neighbours : [np.int64(1), np.int64(6), np.int64(2), np.int64(7), np.int64(3)]
  Distances  : [np.float64(1.3454), np.float64(1.7916), np.float64(2.2361), np.float64(2.3259), np.float64(3.2016)]
  Labels     : ['Yes', 'No', 'No', 'No', 'Yes']
  Votes      : Yes=2, No=3
  ➜ Prediction: No

Student ID 11
  Neighbours : [np.int64(2), np.int64(1), np.int64(3), np.int64(4), np.int64(7)]
  Distances  : [np.float64(1.4142), np.float64(1.7916), np.float64(2.6926), np.float64(3.0414), np.float64(3.1639)]
  Labels     : ['No', 'Yes', 'Yes', 'Yes', 'No']
  Votes      : Yes=3, No=2
  ➜ Prediction: Yes

Student ID 12
  Neighbours : [np.int64(1), np.int64(2), np.

---
## Step 9 — Final Summary Table

In [9]:
print('\n' + '='*50)
print(f'{"ID":<6} {"K=3":^15} {"K=5":^15}')
print('='*50)

for i in range(len(test_ids)):
    sid  = test_ids[i]
    p_k3 = label_map[results_k3[i]['prediction']]
    p_k5 = label_map[results_k5[i]['prediction']]
    flag = '  ← changes!' if p_k3 != p_k5 else ''
    print(f'{sid:<6} {p_k3:^15} {p_k5:^15}{flag}')

print('='*50)


ID           K=3             K=5      
9            No              No       
10           No              No       
11           Yes             Yes      
12           Yes             No         ← changes!
